# Phase 2 specialist track — CheXagent-2 naming (Colab)

Runs the **medical-specialist VLM CheXagent-2 (3B)** on crops from **either V4a (synthetic
tag) or V4b (real chest tube)** under the *exact same blind prompt* the general VLMs
(scout/pixtral) used, writing `predictions_chexagent_<dataset>.jsonl` in the shared file
contract. Set `DATASET` in cell 4 to `'v4a'` or `'v4b'` before running — each writes to its
own filename so one run can't silently overwrite the other.

Score locally with:
- V4a: `python -m auditor.discovery score --manifest manifest.json --predictions predictions_chexagent_v4a.jsonl --confound tag`
- V4b: `python -m auditor.discovery score --manifest manifest.json --predictions predictions_chexagent_v4b.jsonl --confound tube`

**Runtime -> Change runtime type -> T4 GPU** before running.
**Upload** the matching `crops.zip` + `manifest.json` for whichever `DATASET` you set:
- v4a: `experiments/v4a_naming/20260708_105244_da01a4/crops.zip` + `manifest.json`
- v4b: `experiments/v4b_tube/20260708_204951_3894b5/crops.zip` + `manifest.json`

**IMPORTANT — starting a SECOND run in the same or a resumed session (e.g. v4a then v4b):**
`Runtime -> Restart session` first, THEN re-run from cell 1. `!pip install` only changes
what's on disk — if `transformers` was already imported earlier in this Python process (from
a prior run, or Colab's own boot), the OLD version stays cached in memory and the pin in
cell 1 silently does nothing, even though pip reports success. This bit us for real on
2026-07-15: a v4a run succeeded, then a same/resumed-session v4b run hit
`ImportError: cannot import name 'is_tf_available' from 'transformers.utils'` deep inside
CheXagent's own tokenizer code, because Colab's much newer stock `transformers` (which
dropped that TF-availability shim as part of deprecating TensorFlow/Flax support) was still
the one actually loaded. Cell 2 below now asserts the pinned version right after install so
this fails immediately and clearly instead of several cells later inside someone else's code.

**2026-07-15 fix**: the load/inference code below was rewritten against the model card's own
verbatim "Get Started" example (fetched and checked directly, not guessed) after an earlier
version threw persistent errors. It was using the wrong loader entirely (`AutoProcessor`,
a generic vision-processor call), when this model actually needs `AutoTokenizer` +
`tokenizer.from_list_format(...)` + a chat-template conversation — a different calling
convention, not a minor bug. If CheXagent's API has drifted again since, re-check
`https://huggingface.co/StanfordAIMI/CheXagent-2-3b` for the current "Get Started" example —
but the file-contract output format (one `{"id", "description"}` JSON line per crop) and the
PROMPT string (byte-identical to `BLIND_PROMPT` in `src/auditor/discovery/name_groq.py`) must
stay fixed regardless, so specialist-vs-general stays a clean comparison.

In [ ]:
# 1. Dependencies. transformers is PINNED to 4.40.0 -- this model's trust_remote_code=True
# custom code was built/tested against that specific version, and Colab's default (much
# newer) transformers has since dropped internals (e.g. is_tf_available from
# transformers.utils, as TF/Flax backend support was deprecated) that this model's remote
# code still imports -- a likely source of import errors if left unpinned. torch/torchvision
# are left UNPINNED deliberately -- forcing a version reinstall risks replacing Colab's
# CUDA-linked build with a CPU-only wheel; only re-pin those if errors persist.
#
# If this is a SECOND run in this session (e.g. you already ran v4a) -- Runtime -> Restart
# session FIRST. Re-running this cell alone does not undo an already-imported transformers.
!pip -q install "transformers==4.40.0" opencv-python albumentations accelerate pillow \
    matplotlib einops pyarrow sentencepiece protobuf

In [ ]:
# 2. Verify the pin actually took effect BEFORE loading the model -- fails loudly here
# instead of cryptically inside CheXagent's own tokenizer import several cells later.
import transformers
actual = transformers.__version__
assert actual == '4.40.0', (
    f"transformers is {actual}, not the pinned 4.40.0 -- the install in cell 1 did not take "
    "effect (most likely: transformers was already imported earlier in this session). "
    "Runtime -> Restart session, then re-run from cell 1."
)
print('transformers version OK:', actual)

In [ ]:
# 3. Upload crops.zip + manifest.json (the pair matching DATASET below)
from google.colab import files
import zipfile, os
print('Upload crops.zip and manifest.json ...')
up = files.upload()
if 'crops.zip' in up:
    with zipfile.ZipFile('crops.zip') as z:
        z.extractall('.')
assert os.path.isdir('crops'), 'crops/ folder not found after unzip'
assert os.path.exists('manifest.json'), 'manifest.json not found'
print('crops:', len(os.listdir('crops')), '| manifest present')

In [ ]:
# 4. Set which dataset this run is for -> controls the output filename only.
# Upload the MATCHING crops.zip/manifest.json in cell 3 above before running cell 6.
DATASET = 'v4a'  # 'v4a' (synthetic tag) or 'v4b' (real chest tube)
assert DATASET in ('v4a', 'v4b')
OUT_PATH = f'predictions_chexagent_{DATASET}.jsonl'
print('will write ->', OUT_PATH)

In [ ]:
# 5. Load CheXagent-2 (3B) -- AutoTokenizer + AutoModelForCausalLM, bfloat16, device_map='auto'.
# This model does NOT use AutoProcessor (no separate vision processor) -- verified against the
# model card's own "Get Started" example, not the generic vision-model pattern used before.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = 'StanfordAIMI/CheXagent-2-3b'
dtype = torch.bfloat16
device = 'cuda'
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto', trust_remote_code=True)
model = model.to(dtype)
model.eval()
print('model loaded')

In [ ]:
# 6. Blind-prompt loop -> predictions_chexagent_<dataset>.jsonl (same file contract as the
# general VLMs). PROMPT below MUST stay byte-identical to BLIND_PROMPT in
# src/auditor/discovery/name_groq.py -- verified verbatim as of 2026-07-15.
#
# Image+text combination uses tokenizer.from_list_format(...) (a path string, not a PIL
# Image) wrapped in a {"from": ..., "value": ...} conversation and apply_chat_template --
# this model's own documented interface, not a generic processor(images=, text=) call.
import json
PROMPT = (
    'This is a small region that a diagnostic model focused on within a chest X-ray. '
    'Describe anything artificial, non-anatomical, or out of place that you see in this '
    'region. If everything looks like normal anatomy, say so. Be specific and concise.'
)
manifest = json.load(open('manifest.json'))
with open(OUT_PATH, 'w') as f:
    for i, row in enumerate(manifest):
        cid = row['id']
        img_path = f'crops/{cid}.png'
        query = tokenizer.from_list_format([{'image': img_path}, {'text': PROMPT}])
        conv = [{'from': 'system', 'value': 'You are a helpful assistant.'},
                {'from': 'human', 'value': query}]
        input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors='pt')
        output = model.generate(
            input_ids.to(device), do_sample=False, num_beams=1, temperature=1., top_p=1.,
            use_cache=True, max_new_tokens=120,
        )[0]
        # Slice off the input tokens so `description` is only the generated answer, not the
        # echoed prompt -- decoding the full sequence would corrupt the scorer's input.
        desc = tokenizer.decode(output[input_ids.size(1):-1]).strip()
        f.write(json.dumps({'id': cid, 'description': desc}) + '\n')
        if i % 20 == 0:
            print(i, cid, desc[:70])
print('DONE ->', len(manifest), 'predictions at', OUT_PATH)

In [ ]:
# 7. Download for local scoring
from google.colab import files
files.download(OUT_PATH)